# 07 · Ajuste fino QLoRA con Unsloth

**Proyecto:** EmoGol — Trabajo de grado, Universidad del Cauca  
**Autor:** Santiago Fernando Ortega Segura · **Director:** Ph.D. Oscar Mauricio Caicedo Rendón

Ajuste fino de 8 modelos de lenguaje con QLoRA: 4 bits NF4, LoRA r = 16 y α = 16 sobre 7 proyecciones, 3 épocas, tasa de aprendizaje 2×10⁻⁴, lote efectivo 16 y máximo 112 tokens. Guarda checkpoints en Hugging Face y retoma desde el último si la sesión se interrumpe. Esta ejecución corresponde a la sesión final de Llama 3.1 8B; los demás modelos usan el mismo notebook cambiando `CLAVE_MODELO`.

*Nombre durante el desarrollo: `06_entrenamiento_real_unsloth` (algunas salidas conservan esa numeración).*

## 1. Instalar dependencias

In [1]:
!pip install -U transformers bitsandbytes unsloth unsloth_zoo huggingface_hub -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.2/81.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 121.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/

## 2. Configuración

In [2]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

TU_USUARIO_HF = "santiagortegasegura"

MODELOS_DISPONIBLES = {
    "qwen_1_5b":  "Qwen/Qwen2.5-1.5B-Instruct",
    "llama_3b":   "meta-llama/Llama-3.2-3B-Instruct",
    "qwen_3b":    "Qwen/Qwen2.5-3B-Instruct",
    "phi35_mini": "microsoft/Phi-3.5-mini-instruct",
    "phi3_mini":  "microsoft/Phi-3-mini-4k-instruct",
    "mistral_7b": "mistralai/Mistral-7B-Instruct-v0.3",
    "qwen_7b":    "Qwen/Qwen2.5-7B-Instruct",
    "llama_8b":   "meta-llama/Meta-Llama-3.1-8B-Instruct",
}
CLAVE_MODELO = "llama_8b"
MODEL_ID = MODELOS_DISPONIBLES[CLAVE_MODELO]

REPO_DATOS = f"{TU_USUARIO_HF}/emotion-futbol-data"
REPO_CHECKPOINTS = f"{TU_USUARIO_HF}/emotion-futbol-checkpoints"

COLUMNA_TEXTO = "text"
MAX_LENGTH = 112
EPOCAS = 3

print(f"Modelo a entrenar: {CLAVE_MODELO} -> {MODEL_ID}")

Modelo a entrenar: llama_8b -> meta-llama/Meta-Llama-3.1-8B-Instruct


## 3. Revisar si el modelo ya está entrenado

In [3]:
from huggingface_hub import hf_hub_download

YA_COMPLETO = True
try:
    hf_hub_download(repo_id=REPO_CHECKPOINTS, repo_type="dataset",
                     filename=f"{CLAVE_MODELO}/ENTRENAMIENTO_COMPLETO.txt")
except Exception:
    YA_COMPLETO = False

if YA_COMPLETO:
    raise SystemExit(f"'{CLAVE_MODELO}' ya está completamente entrenado (ENTRENAMIENTO_COMPLETO.txt "
                      f"ya existe en tu bodega). No hace falta correr nada más para este modelo -- "
                      f"cambia CLAVE_MODELO al siguiente en la Sección 2.")
else:
    print(f"'{CLAVE_MODELO}' todavía no está completo -- seguimos.")

'llama_8b' todavía no está completo -- seguimos.


## 4. Buscar un checkpoint previo

In [4]:
import glob
from huggingface_hub import snapshot_download

LOCAL_CKPT_DIR = f"/content/ckpt_{CLAVE_MODELO}"

try:
    snapshot_download(
        repo_id=REPO_CHECKPOINTS, repo_type="dataset",
        allow_patterns=f"{CLAVE_MODELO}/checkpoint-*/*",
        local_dir=LOCAL_CKPT_DIR,
    )
except Exception as e:
    print("No se encontraron checkpoints previos (o aún no existen):", e)

candidatos = glob.glob(f"{LOCAL_CKPT_DIR}/{CLAVE_MODELO}/checkpoint-*")
RESUME_FROM = None
if candidatos:
    candidatos.sort(key=lambda p: int(p.rstrip("/").split("-")[-1]))
    RESUME_FROM = candidatos[-1]
    print(f"Se encontró un checkpoint previo -- retomando desde: {RESUME_FROM}")
else:
    print("No hay checkpoints previos -- este modelo empieza desde cero.")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Se encontró un checkpoint previo -- retomando desde: /content/ckpt_llama_8b/llama_8b/checkpoint-13200


## 5. Descargar los datos

In [5]:
import pandas as pd

ruta_train = hf_hub_download(repo_id=REPO_DATOS, repo_type="dataset",
                              filename="04b_entrenamiento_72k.csv")
ruta_val_interna = hf_hub_download(repo_id=REPO_DATOS, repo_type="dataset",
                                    filename="04b_validacion_interna_18k.csv")

df_entrenamiento = pd.read_csv(ruta_train)
df_validacion_interna = pd.read_csv(ruta_val_interna)

print(f"Entrenamiento: {len(df_entrenamiento)} ejemplos")
print(f"Validación interna: {len(df_validacion_interna)} ejemplos")

04b_entrenamiento_72k.csv:   0%|          | 0.00/8.65M [00:00<?, ?B/s]

04b_validacion_interna_18k.csv:   0%|          | 0.00/2.16M [00:00<?, ?B/s]

Entrenamiento: 72000 ejemplos
Validación interna: 18000 ejemplos


## 6. Cargar el modelo (4-bit, LoRA) con Unsloth

In [6]:
from unsloth import FastLanguageModel
import torch

torch.cuda.reset_peak_memory_stats()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_ID,
    max_seq_length = MAX_LENGTH,
    load_in_4bit = True,
    dtype = None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

print("Modelo y LoRA cargados sin errores.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.9.7 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Modelo y LoRA cargados sin errores.


## 7. Preparar los datasets con el formato de prompt

In [7]:
EMOCIONES = ['alegria', 'tristeza', 'enojo', 'miedo', 'sorpresa', 'burla', 'neutral']

def formatear_prompt(row):
    return (
        f"Clasifica la emoción del siguiente comentario de fútbol en una de estas categorías: "
        f"{', '.join(EMOCIONES)}.\n\nComentario: {row[COLUMNA_TEXTO]}\n\nEmoción: {row['label']}"
    )

df_entrenamiento['texto_formateado'] = df_entrenamiento.apply(formatear_prompt, axis=1)

df_validacion_interna['texto_formateado'] = df_validacion_interna.apply(formatear_prompt, axis=1)

from datasets import Dataset
dataset_entrenamiento = Dataset.from_pandas(df_entrenamiento[['texto_formateado']])
dataset_eval = Dataset.from_pandas(df_validacion_interna[['texto_formateado']])

print(f"Entrenamiento: {len(dataset_entrenamiento)} ejemplos | Evaluación (18k completas): {len(dataset_eval)} ejemplos")

Entrenamiento: 72000 ejemplos | Evaluación (18k completas): 18000 ejemplos


## 8. Entrenar (con checkpoints en Hugging Face)

In [8]:
import time
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
from huggingface_hub import upload_folder, HfApi

api_hf = HfApi()

MINUTOS_ENTRE_GUARDADOS = 20

class GuardarPorTiempoCallback(TrainerCallback):
    def __init__(self, minutos=MINUTOS_ENTRE_GUARDADOS):
        self.intervalo_seg = minutos * 60
        self.ultimo_guardado = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        ahora = time.time()
        if control.should_save:
            self.ultimo_guardado = ahora
        elif ahora - self.ultimo_guardado >= self.intervalo_seg:
            control.should_save = True
            self.ultimo_guardado = ahora
        return control

SUBIDAS_ENTRE_SQUASH = 8
class SubirCheckpointCallback(TrainerCallback):
    def __init__(self):
        self.subidas_desde_squash = 0

    def on_save(self, args, state, control, **kwargs):
        paso_actual = state.global_step
        ckpt_dir = f"{args.output_dir}/checkpoint-{paso_actual}"
        try:
            upload_folder(
                folder_path=ckpt_dir,
                repo_id=REPO_CHECKPOINTS,
                repo_type="dataset",
                path_in_repo=f"{CLAVE_MODELO}/checkpoint-{paso_actual}",
            )
            print(f"  -> Checkpoint del paso {paso_actual} subido a Hugging Face.")
        except Exception as e:
            print(f"  -> No se pudo subir el checkpoint del paso {paso_actual} "
                  f"(sigue entrenando de todas formas): {e}")
            return

        try:
            archivos = api_hf.list_repo_files(repo_id=REPO_CHECKPOINTS, repo_type="dataset")
            prefijo = f"{CLAVE_MODELO}/checkpoint-"
            pasos_viejos = {
                int(a[len(prefijo):].split("/")[0])
                for a in archivos
                if a.startswith(prefijo) and a[len(prefijo):].split("/")[0].isdigit()
                and int(a[len(prefijo):].split("/")[0]) != paso_actual
            }
            for paso_viejo in pasos_viejos:
                api_hf.delete_folder(repo_id=REPO_CHECKPOINTS, repo_type="dataset",
                                      path_in_repo=f"{CLAVE_MODELO}/checkpoint-{paso_viejo}")
                print(f"  -> Checkpoint viejo del paso {paso_viejo} borrado de Hugging Face "
                      f"(para no acumular espacio ni hacer más lento el próximo 'retomar').")
        except Exception as e:
            print(f"  -> No se pudo limpiar checkpoints viejos (no es grave, solo ocupa más espacio): {e}")

        self.subidas_desde_squash += 1
        if self.subidas_desde_squash >= SUBIDAS_ENTRE_SQUASH:
            try:
                api_hf.super_squash_history(repo_id=REPO_CHECKPOINTS, repo_type="dataset")
                print("  -> Historial de la bodega comprimido (libera el espacio de los "
                      "checkpoints viejos de verdad, no solo del árbol actual).")
                self.subidas_desde_squash = 0
            except Exception as e:
                print(f"  -> No se pudo comprimir el historial todavía (no es grave, se "
                      f"reintenta en el próximo guardado): {e}")

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset_entrenamiento,
    eval_dataset = dataset_eval,
    args = SFTConfig(
        dataset_text_field = "texto_formateado",
        max_length = MAX_LENGTH,
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 2,
        num_train_epochs = EPOCAS,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 25,
        eval_strategy = "epoch",
        save_strategy = "steps",
        save_steps = 200,
        save_total_limit = 3,
        output_dir = f"{LOCAL_CKPT_DIR}/{CLAVE_MODELO}",
    ),
)
trainer.add_callback(GuardarPorTiempoCallback())
trainer.add_callback(SubirCheckpointCallback())

trainer.train(resume_from_checkpoint=RESUME_FROM)

print(f"\nPico de VRAM usada durante todo el entrenamiento: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

Unsloth: Tokenizing ["texto_formateado"] (num_proc=2):   0%|          | 0/72000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["texto_formateado"] (num_proc=2):   0%|          | 0/18000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 72,000 | Num Epochs = 3 | Total steps = 13,500
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-

Epoch,Training Loss,Validation Loss
3,0.666668,0.890387


Unsloth: Restored added_tokens_decoder metadata in /content/ckpt_llama_8b/llama_8b/checkpoint-13400/tokenizer_config.json.


  -> Checkpoint del paso 13400 subido a Hugging Face.
  -> Checkpoint viejo del paso 13200 borrado de Hugging Face (para no acumular espacio ni hacer más lento el próximo 'retomar').


Unsloth: Restored added_tokens_decoder metadata in /content/ckpt_llama_8b/llama_8b/checkpoint-13500/tokenizer_config.json.


  -> Checkpoint del paso 13500 subido a Hugging Face.
  -> Checkpoint viejo del paso 13400 borrado de Hugging Face (para no acumular espacio ni hacer más lento el próximo 'retomar').

Pico de VRAM usada durante todo el entrenamiento: 7.21 GB


## 9. Guardar y subir el adaptador final

In [9]:
from huggingface_hub import upload_file

CARPETA_FINAL = f"{LOCAL_CKPT_DIR}/{CLAVE_MODELO}/adapter_final"
model.save_pretrained(CARPETA_FINAL)
tokenizer.save_pretrained(CARPETA_FINAL)

upload_folder(
    folder_path=CARPETA_FINAL,
    repo_id=REPO_CHECKPOINTS,
    repo_type="dataset",
    path_in_repo=f"{CLAVE_MODELO}/adapter_final",
)

with open("/content/marca_completo.txt", "w") as f:
    f.write(f"Entrenamiento completo: {CLAVE_MODELO} ({MODEL_ID}), {EPOCAS} épocas sobre 72k.\n")

upload_file(
    path_or_fileobj="/content/marca_completo.txt",
    path_in_repo=f"{CLAVE_MODELO}/ENTRENAMIENTO_COMPLETO.txt",
    repo_id=REPO_CHECKPOINTS,
    repo_type="dataset",
)

try:
    archivos = api_hf.list_repo_files(repo_id=REPO_CHECKPOINTS, repo_type="dataset")
    prefijo = f"{CLAVE_MODELO}/checkpoint-"
    pasos_a_borrar = {a[len(prefijo):].split("/")[0] for a in archivos if a.startswith(prefijo)}
    for paso in pasos_a_borrar:
        api_hf.delete_folder(repo_id=REPO_CHECKPOINTS, repo_type="dataset",
                              path_in_repo=f"{CLAVE_MODELO}/checkpoint-{paso}")
    if pasos_a_borrar:
        print(f"Checkpoints intermedios de '{CLAVE_MODELO}' borrados (ya no hacen falta, quedó el adapter_final).")
except Exception as e:
    print(f"No se pudieron borrar los checkpoints intermedios (no es grave): {e}")

try:
    api_hf.super_squash_history(repo_id=REPO_CHECKPOINTS, repo_type="dataset")
    print("Historial de la bodega comprimido -- el espacio de los checkpoints intermedios ya quedó liberado de verdad.")
except Exception as e:
    print(f"No se pudo comprimir el historial ahora (no es grave, puedes correr esto luego a mano): {e}")

print(f"\nListo -- '{CLAVE_MODELO}' entrenado y guardado en tu bodega de Hugging Face.")
print("Cambia CLAVE_MODELO en la Sección 2 al siguiente modelo de la lista y vuelve a correr todo.")

Unsloth: Restored added_tokens_decoder metadata in /content/ckpt_llama_8b/llama_8b/adapter_final/tokenizer_config.json.


Checkpoints intermedios de 'llama_8b' borrados (ya no hacen falta, quedó el adapter_final).
Historial de la bodega comprimido -- el espacio de los checkpoints intermedios ya quedó liberado de verdad.

Listo -- 'llama_8b' entrenado y guardado en tu bodega de Hugging Face.
Cambia CLAVE_MODELO en la Sección 2 al siguiente modelo de la lista y vuelve a correr todo.
